# 09 - Build Analytical Tables

This notebook builds the three analytical master tables used in the portfolio analysis.

- `destinations_master.csv` supports Q1.
- `entry_reasons_master.csv` supports Q2.
- `post_arrival_master.csv` supports Q3.

The notebook starts from standardized files only. It does not modify raw data.


## Methodological Rules

The analytical tables keep different indicators separated. Stocks, inflows, permits, long-term residents, status changes and citizenship acquisitions are not added together as if they measured the same phenomenon.


In [1]:
from pathlib import Path

import pandas as pd


In [2]:
PROJECT_ROOT = Path('..')

STANDARDIZED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'standardized'
ANALYTICAL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'analytical'

ANALYTICAL_PATH.mkdir(parents=True, exist_ok=True)


## Load Standardized Files

Each standardized file already contains harmonized columns such as `source`, `dataset`, `origin_country`, `destination_country`, `year`, `period`, `measure_type` and `value`.


In [3]:
undesa = pd.read_csv(STANDARDIZED_PATH / 'undesa_destinations_standardized.csv')
eurostat_first = pd.read_csv(STANDARDIZED_PATH / 'eurostat_first_permits_standardized.csv')
eurostat_long = pd.read_csv(STANDARDIZED_PATH / 'eurostat_long_term_residents_standardized.csv')
eurostat_status = pd.read_csv(STANDARDIZED_PATH / 'eurostat_status_changes_standardized.csv')
eurostat_citizenship = pd.read_csv(STANDARDIZED_PATH / 'eurostat_citizenship_acquisition_standardized.csv')
oecd = pd.read_csv(STANDARDIZED_PATH / 'oecd_migration_standardized.csv')


In [4]:
def keep_columns(df, columns):
    """Keep a clear column order while allowing optional columns to be absent."""
    return df[[col for col in columns if col in df.columns]].copy()


def ensure_sub_reason(df, default_value='not_applicable'):
    """Guarantee a stable Q2 schema for sources with or without sub-reasons."""
    df = df.copy()
    if 'sub_reason' not in df.columns:
        df['sub_reason'] = default_value
    else:
        missing = df['sub_reason'].isna() | (df['sub_reason'].astype('string').str.strip() == '')
        df.loc[missing, 'sub_reason'] = default_value
    return df


## Q1 - Destination Countries

UN DESA is the main global stock source. OECD destination indicators are kept as complementary evidence and remain separated by `measure_type`.


In [5]:
oecd_destinations = oecd[oecd['analysis_question'].eq('Q1_destinations')].copy()

destinations_master = pd.concat([undesa, oecd_destinations], ignore_index=True)

destination_cols = [
    'source',
    'dataset',
    'origin_country',
    'destination_country',
    'year',
    'period',
    'measure_type',
    'value',
    'total_migrants',
    'male_migrants',
    'female_migrants',
    'analysis_question',
]

destinations_master = keep_columns(destinations_master, destination_cols)
destinations_master.head()


,source,dataset,origin_country,destination_country,year,period,measure_type,value,total_migrants,male_migrants,female_migrants,analysis_question
0,undesa,undesa_destinations,Cameroon,Zambia,2015,pre_covid,stock,214.0,214.0,114.0,100.0,Q1_destinations
1,undesa,undesa_destinations,Cameroon,Chad,2015,pre_covid,stock,30702.0,30702.0,13908.0,16794.0,Q1_destinations
2,undesa,undesa_destinations,Cameroon,Congo,2015,pre_covid,stock,11300.0,11300.0,6727.0,4573.0,Q1_destinations
3,undesa,undesa_destinations,Cameroon,Equatorial Guinea,2015,pre_covid,stock,798.0,798.0,535.0,263.0,Q1_destinations
4,undesa,undesa_destinations,Cameroon,Gabon,2015,pre_covid,stock,46269.0,46269.0,26905.0,19364.0,Q1_destinations


In [6]:
print('Shape:', destinations_master.shape)
print('Years:', sorted(destinations_master['year'].dropna().unique()))
print('Sources:', destinations_master['source'].dropna().unique())
print('Measure types:', destinations_master['measure_type'].dropna().unique())


Shape: (695, 12)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2024)]
Sources: <StringArray>
['undesa', 'oecd']
Length: 2, dtype: str
Measure types: <StringArray>
['stock', 'inflow', 'stock_birth', 'stock_nationality']
Length: 4, dtype: str


## Q2 - Observable Reasons for Entry

Eurostat first permits are the main source for entry reasons. OECD worker, seasonal worker and asylum inflows are complementary indicators. They are not merged into Eurostat permit totals.


In [7]:
eurostat_first = ensure_sub_reason(eurostat_first, default_value='not_applicable')
oecd_entry_reasons = ensure_sub_reason(
    oecd[oecd['analysis_question'].eq('Q2_entry_reasons')].copy(),
    default_value='not_applicable',
)

entry_reasons_master = pd.concat([eurostat_first, oecd_entry_reasons], ignore_index=True)

entry_cols = [
    'source',
    'dataset',
    'origin_country',
    'destination_country',
    'year',
    'period',
    'reason',
    'sub_reason',
    'measure_type',
    'value',
    'analysis_question',
]

entry_reasons_master = keep_columns(entry_reasons_master, entry_cols)
entry_reasons_master.head()


,source,dataset,origin_country,destination_country,year,period,reason,sub_reason,measure_type,value,analysis_question
0,eurostat,eurostat_first_permits,Cameroon,Belgium,2015,pre_covid,family,not_applicable,permit,695.0,Q2_entry_reasons
1,eurostat,eurostat_first_permits,Cameroon,Bulgaria,2015,pre_covid,family,not_applicable,permit,0.0,Q2_entry_reasons
2,eurostat,eurostat_first_permits,Cameroon,Czechia,2015,pre_covid,family,not_applicable,permit,15.0,Q2_entry_reasons
3,eurostat,eurostat_first_permits,Cameroon,Denmark,2015,pre_covid,family,not_applicable,permit,55.0,Q2_entry_reasons
4,eurostat,eurostat_first_permits,Cameroon,Germany,2015,pre_covid,family,not_applicable,permit,748.0,Q2_entry_reasons


In [8]:
print('Shape:', entry_reasons_master.shape)
print('Years:', sorted(entry_reasons_master['year'].dropna().unique()))
print('Sources:', entry_reasons_master['source'].dropna().unique())
print('Measure types:', entry_reasons_master['measure_type'].dropna().unique())
print('Sub-reasons:', entry_reasons_master['sub_reason'].dropna().unique())


Shape: (1595, 11)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Sources: <StringArray>
['eurostat', 'oecd']
Length: 2, dtype: str
Measure types: <StringArray>
['permit', 'asylum_inflow', 'worker_inflow', 'seasonal_worker_inflow']
Length: 4, dtype: str
Sub-reasons: <StringArray>
['not_applicable', 'asylum', 'work', 'seasonal_work']
Length: 4, dtype: str


## Q3 - Post-Arrival Trajectories

Eurostat provides the main Q3 indicators for long-term residence, status changes and citizenship acquisition. OECD citizenship acquisition remains in the master table as complementary evidence, but the exploratory notebook analyzes Eurostat and OECD separately.


In [9]:
oecd_post_arrival = oecd[oecd['analysis_question'].eq('Q3_post_arrival_trajectories')].copy()

post_arrival_master = pd.concat(
    [eurostat_long, eurostat_status, eurostat_citizenship, oecd_post_arrival],
    ignore_index=True,
)

post_arrival_cols = [
    'source',
    'dataset',
    'origin_country',
    'destination_country',
    'year',
    'period',
    'measure_type',
    'legal_framework',
    'value',
    'analysis_question',
]

post_arrival_master = keep_columns(post_arrival_master, post_arrival_cols)
post_arrival_master.head()


,source,dataset,origin_country,destination_country,year,period,measure_type,value,analysis_question
0,eurostat,eurostat_long_term_residents,Cameroon,Belgium,2015,pre_covid,long_term_resident,977.0,Q3_post_arrival_trajectories
1,eurostat,eurostat_long_term_residents,Cameroon,Bulgaria,2015,pre_covid,long_term_resident,6.0,Q3_post_arrival_trajectories
2,eurostat,eurostat_long_term_residents,Cameroon,Czechia,2015,pre_covid,long_term_resident,53.0,Q3_post_arrival_trajectories
3,eurostat,eurostat_long_term_residents,Cameroon,Denmark,2015,pre_covid,long_term_resident,121.0,Q3_post_arrival_trajectories
4,eurostat,eurostat_long_term_residents,Cameroon,Germany,2015,pre_covid,long_term_resident,80.0,Q3_post_arrival_trajectories


In [10]:
print('Shape:', post_arrival_master.shape)
print('Years:', sorted(post_arrival_master['year'].dropna().unique()))
print('Sources:', post_arrival_master['source'].dropna().unique())
print('Measure types:', post_arrival_master['measure_type'].dropna().unique())


Shape: (948, 9)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Sources: <StringArray>
['eurostat', 'oecd']
Length: 2, dtype: str
Measure types: <StringArray>
['long_term_resident', 'status_change', 'citizenship_acquisition']
Length: 3, dtype: str


## Export Analytical Tables

These files are the stable inputs for exploratory analysis, reporting tables and final visualizations.


In [11]:
destinations_master.to_csv(ANALYTICAL_PATH / 'destinations_master.csv', index=False)
entry_reasons_master.to_csv(ANALYTICAL_PATH / 'entry_reasons_master.csv', index=False)
post_arrival_master.to_csv(ANALYTICAL_PATH / 'post_arrival_master.csv', index=False)

print('Analytical tables exported successfully.')


Analytical tables exported successfully.


## Summary

- `destinations_master.csv` is used for Q1 destination analysis.
- `entry_reasons_master.csv` is used for Q2 entry reason analysis and keeps `sub_reason`.
- `post_arrival_master.csv` is used for Q3 post-arrival analysis, with Eurostat and OECD kept identifiable by source.
